In [1]:
!pip install optuna


   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   ---------------------------------------- 2/2 [optuna]



In [2]:
import xgboost as xgb

from sklearn.metrics import roc_auc_score

print("XGBoost Loaded ✅")

XGBoost Loaded ✅


In [5]:
import optuna, mlflow
from sklearn.model_selection import cross_val_score

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'max_depth':    trial.suggest_int('max_depth', 3, 10),
        'learning_rate':trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':    trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':        trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':    trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda':   trial.suggest_float('reg_lambda', 0.0, 2.0),
    }
    model = xgb.XGBClassifier(**params, eval_metric='auc',
                              scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(), random_state=42)
    score = cross_val_score(model, X_tr, y_tr, cv=4, scoring='roc_auc').mean()
    return score

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(xgb_objective, n_trials=60, show_progress_bar=True)

print(f"Best AUC (CV): {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

# Refit best, evaluate on test
best = xgb.XGBClassifier(**study.best_params, eval_metric='auc', random_state=42)
best.fit(X_tr, y_tr)
final_auc = roc_auc_score(y_te, best.predict_proba(X_te)[:,1])
print(f"Final test AUC: {final_auc:.4f}")

[I 2026-05-21 15:22:21,657] A new study created in memory with name: no-name-75fca851-647a-4009-8b53-50512f96253b


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-05-21 15:22:24,679] Trial 0 finished with value: 0.9453191179424425 and parameters: {'n_estimators': 425, 'max_depth': 10, 'learning_rate': 0.08960785365368121, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'gamma': 0.2904180608409973, 'reg_alpha': 1.7323522915498704, 'reg_lambda': 1.2022300234864176}. Best is trial 0 with value: 0.9453191179424425.
[I 2026-05-21 15:22:26,562] Trial 1 finished with value: 0.9455753444090418 and parameters: {'n_estimators': 625, 'max_depth': 3, 'learning_rate': 0.18276027831785724, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'min_child_weight': 2, 'gamma': 0.9170225492671691, 'reg_alpha': 0.6084844859190754, 'reg_lambda': 1.0495128632644757}. Best is trial 1 with value: 0.9455753444090418.
[I 2026-05-21 15:22:28,169] Trial 2 finished with value: 0.9431263669083677 and parameters: {'n_estimators': 459, 'max_depth': 5, 'learning_rate': 0.06252287916406217, 'subsample': 

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# Load datasets

rfm = pd.read_csv(

    "data/processed/rfm_features.csv"

)

churn = pd.read_csv(

    "data/raw/churn_labels.csv"

)

sales = pd.read_csv(

    "data/processed/sales_cleaned.csv",

    parse_dates=['InvoiceDate']

)

# ===== RECENT 90 DAYS FEATURES =====

recent = sales[

    sales['InvoiceDate']

    >=

    sales['InvoiceDate'].max()

    - pd.Timedelta(days=90)

]

recent_agg = recent.groupby(

    'CustomerID'

).agg(

    recent_orders=('Invoice', 'nunique'),

    recent_spend=('TotalPrice', 'sum'),

    recent_aov=('TotalPrice', 'mean')

).reset_index()

# ===== TYPE FIX =====

recent_agg['CustomerID'] = (

    recent_agg['CustomerID']

    .astype(str)

)

rfm['CustomerID'] = (

    rfm['CustomerID']

    .astype(str)

)

churn['CustomerID'] = (

    churn['CustomerID']

    .astype(str)

)

# ===== MERGE =====

df = rfm.merge(

    recent_agg,

    on='CustomerID',

    how='left'

).fillna(0)

df = df.merge(

    churn[['CustomerID', 'Churned']],

    on='CustomerID'

)

# ===== FEATURES =====

feat_cols = [

    'Recency',

    'Frequency',

    'Monetary',

    'AOV',

    'UniqueProducts',

    'UniqueCategories',

    'Tenure',

    'AvgQty',

    'recent_orders',

    'recent_spend',

    'recent_aov',

    'R_Score',

    'F_Score',

    'M_Score'

]

# ===== ENCODE CATEGORICALS =====

for c in [

    'AgeGroup',

    'Gender',

    'LoyaltyTier',

    'PreferredChannel',

    'Country'

]:

    df[c] = (

        df[c]

        .astype('category')

        .cat.codes

    )

    feat_cols.append(c)

# ===== TRAIN TEST SPLIT =====

X = df[feat_cols]

y = df['Churned']

X_tr, X_te, y_tr, y_te = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

print("Train Shape:", X_tr.shape)

print("Test Shape :", X_te.shape)

print("Variables Ready ✅")

Train Shape: (6056, 19)
Test Shape : (1515, 19)
Variables Ready ✅


In [4]:
print(X_tr.shape)

print(y_tr.shape)

(6056, 19)
(6056,)
